# Creating Specialties Dimensions

This script creates a one row per NPI with the most recent specialty information of our data selections.

This script relies on the DMEPOS Referring Provider and General Payments for specialty information. 

This script requires /dsa/groups/casestudycf25/team02/general_payments_providers_clean.csv and /dsa/groups/casestudycf25/team02/DMEPOS_rfrr_clean.csv.

Data Cleaning
- Keep only one row per NPI from the DMEPOS dataset, prioritizing NPPES sourced and most recent data
- Keep most recent program year specialty information first from General Payments provider csv

It returns ("dmepos_specialties_unique.csv", "gp_specialties_unique.csv") as interim CSVs.

It *requires* a manually created CSV called specialty_crosswalk_gpt_backfill.csv
This manual CSV was created by using the outputs of the dmepos_specialties_unique and the gp_specialties_unique CSVs for manual deduplication and backfill of specialty hierarchy information. ChatGPT was used to fill out the groups and then manual validation was applied to the groupings. This is then merged to the one_specialty_per_npi_final by the og_specialty_name to get the hierarchy decriptions. 

It returns the final one row per NPI with specialty information dataset: /dsa/groups/casestudycf25/team02/one_specialty_per_npi.csv 

In [1]:
import pandas as pd

### Using the DMEPOS Referring Providers Dataset

In [2]:
dme_year_npi = pd.read_csv('/dsa/groups/casestudycf25/team02/DMEPOS_rfrr_clean.csv',dtype={'Rfrg_Prvdr_State_FIPS':str,'Rfrg_Prvdr_Zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str


In [10]:
dme_specialties = dme_year_npi[["Rfrg_Prvdr_Spclty_Desc"]].copy()
dme_specialties.drop_duplicates(inplace = True, keep="first")
dme_specialties

,Rfrg_Prvdr_Spclty_Desc
0,Internal Medicine
1,General Surgery
2,Family Practice
4,Urology
6,Cardiology
...,...
864619,Clinical Neuropsychologist
888244,Home Health
896675,Rehabilitation Counselor
925565,Integrative Medicine


In [11]:
dme_specialties.to_csv("dmepos_specialties_unique.csv")

In [3]:
dme_specialties_npi = dme_year_npi[["Rfrg_NPI","Rfrg_Prvdr_Spclty_Desc", "Rfrg_Prvdr_Spclty_Srce" , "Year"]].copy()

In [4]:
# Filter by NPPES Specialty derived rows
dme_specialties_nppes = dme_specialties_npi[dme_specialties_npi["Rfrg_Prvdr_Spclty_Srce"] == "NPPES-Specialty"]
dme_specialties_nppes = dme_specialties_nppes.sort_values('Year', ascending=False)

# Keep the first and most recent specialty information
dme_specialties_nppes.drop_duplicates(subset = ["Rfrg_NPI"] , inplace = True, keep="first")

dme_specialties_nppes.drop(columns = "Year", inplace = True)

In [5]:
# Filter by Claim Specialty derived rows
dme_specialties_claims = dme_specialties_npi[dme_specialties_npi["Rfrg_Prvdr_Spclty_Srce"] == "Claim-Specialty"]
dme_specialties_claims = dme_specialties_claims.sort_values('Year', ascending=False)

# Keep the first and most recent specialty information
dme_specialties_claims.drop_duplicates(subset = ["Rfrg_NPI"] , inplace = True, keep="first")

dme_specialties_claims.drop(columns = "Year", inplace = True)

In [6]:
# Append the specialties subsets, with the NPPES sourced rows first
temp_append = pd.concat([dme_specialties_nppes, dme_specialties_claims], ignore_index=True)

In [7]:
# Sort it by the source to prioritize NPPES specialty information first
dme_specialties_npi_sorted = temp_append.sort_values('Rfrg_Prvdr_Spclty_Srce', ascending=False)

# Keep only one row per NPI from the DMEPOS dataset, prioritizing NPPES sourced and most recent info 
dme_specialties_npi_sorted.drop_duplicates(subset = ["Rfrg_NPI"] , inplace = True, keep="first")

In [8]:
dme_specialties_npi_sorted

,Rfrg_NPI,Rfrg_Prvdr_Spclty_Desc,Rfrg_Prvdr_Spclty_Srce
0,1992997787,Physician Assistant,NPPES-Specialty
36092,1144674441,Nurse Practitioner,NPPES-Specialty
36068,1144740838,General Surgery,NPPES-Specialty
36069,1144650235,Nurse Practitioner,NPPES-Specialty
36070,1144648395,Nurse Practitioner,NPPES-Specialty
...,...,...,...
206795,1235197088,Plastic and Reconstructive Surgery,Claim-Specialty
206794,1235197211,Internal Medicine,Claim-Specialty
206793,1235197237,Nurse Practitioner,Claim-Specialty
206792,1235198342,Endocrinology,Claim-Specialty


In [9]:
del dme_specialties_npi , dme_specialties_nppes, dme_specialties_claims

### General Payments Specialties

In [12]:
gp_providers = pd.read_csv("/dsa/groups/casestudycf25/team02/general_payments_providers_clean.csv")

/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3058: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [14]:
gp_providers.head()

,Unnamed: 0,covered_recipient_profile_id,covered_recipient_npi,covered_recipient_type,covered_recipient_primary_type_1,covered_recipient_specialty_1,specialty_lvl1,specialty_lvl2,specialty_lvl3,covered_recipient_license_state_code1,...,covered_recipient_license_state_code4,covered_recipient_license_state_code5,recipient_primary_business_street_address_line1,recipient_primary_business_street_address_line2,recipient_city,recipient_state,recipient_zip_code,recipient_country,recipient_postal_code,program_year
0,0,294069,1497734156,Covered Recipient Physician,Medical Doctor,Allopathic & Osteopathic Physicians|Orthopaedi...,Allopathic & Osteopathic Physicians,Orthopaedic Surgery,NaN,NC,...,NaN,NaN,2001 VAIL AVE,SUITE 200B,CHARLOTTE,NC,28207.0,United States,NaN,2023
1,4,11322702,1932840923,Covered Recipient Physician,Medical Doctor,Allopathic & Osteopathic Physicians|Orthopaedi...,Allopathic & Osteopathic Physicians,Orthopaedic Surgery,NaN,MI,...,NaN,NaN,221 MICHIGAN ST NE STE 400,NaN,GRAND RAPIDS,MI,49503.0,United States,NaN,2023
2,5,434829,1518070135,Covered Recipient Physician,Medical Doctor,Allopathic & Osteopathic Physicians|Otolaryngo...,Allopathic & Osteopathic Physicians,Otolaryngology,NaN,SC,...,NaN,NaN,9228 MEDICAL PLAZA DR,NaN,CHARLESTON,SC,29406.0,United States,NaN,2023
3,6,552503,1649439860,Covered Recipient Physician,Medical Doctor,Allopathic & Osteopathic Physicians|Orthopaedi...,Allopathic & Osteopathic Physicians,Orthopaedic Surgery,Sports Medicine,PA,...,NaN,NaN,170 N POINTE BLVD,NaN,LANCASTER,PA,17601.0,United States,NaN,2023
4,7,11452165,1750964185,Covered Recipient Physician,Medical Doctor,Allopathic & Osteopathic Physicians|Orthopaedi...,Allopathic & Osteopathic Physicians,Orthopaedic Surgery,NaN,IL,...,NaN,NaN,600 S PAULINA ST,NaN,CHICAGO,IL,60612.0,United States,NaN,2023


In [19]:
gp_specialties = gp_providers[["covered_recipient_primary_type_1", "specialty_lvl1", "specialty_lvl2"]].copy()
gp_specialties.drop_duplicates(subset = 'specialty_lvl2', inplace = True, keep="first")
gp_specialties

,covered_recipient_primary_type_1,specialty_lvl1,specialty_lvl2
0,Medical Doctor,Allopathic & Osteopathic Physicians,Orthopaedic Surgery
2,Medical Doctor,Allopathic & Osteopathic Physicians,Otolaryngology
7,Doctor of Podiatric Medicine,Podiatric Medicine & Surgery Service Providers,Podiatrist
22,Medical Doctor,Allopathic & Osteopathic Physicians,General Practice
25,Doctor of Dentistry,Dental Providers,Dentist
26,Medical Doctor,Allopathic & Osteopathic Physicians,Internal Medicine
32,Medical Doctor,Allopathic & Osteopathic Physicians,Neurological Surgery
45,Medical Doctor,Allopathic & Osteopathic Physicians,Anesthesiology
67,Medical Doctor,Allopathic & Osteopathic Physicians,Surgery
104,Medical Doctor,Allopathic & Osteopathic Physicians,Psychiatry & Neurology


In [20]:
gp_specialties.to_csv("gp_specialties_unique.csv")

In [21]:
gp_specialties_npi = gp_providers[["covered_recipient_npi","specialty_lvl2", "program_year"]].copy()

In [22]:
gp_specialties_npi = gp_specialties_npi.sort_values('program_year', ascending=False)

# Keep most recent program year specialty information first
gp_specialties_npi.drop_duplicates(subset = ["covered_recipient_npi"] , inplace = True, keep="first")

gp_specialties_npi.drop(columns = "program_year", inplace = True)

In [24]:
gp_specialties_npi["Rfrg_Prvdr_Spclty_Srce"] = "Payments-Specialty"

In [28]:
gp_specialties_npi

,covered_recipient_npi,specialty_lvl2,Rfrg_Prvdr_Spclty_Srce
0,1497734156,Orthopaedic Surgery,Payments-Specialty
799481,1275536377,Family Medicine,Payments-Specialty
799455,1609893270,Internal Medicine,Payments-Specialty
799456,1669440822,Internal Medicine,Payments-Specialty
799457,1003815317,Obstetrics & Gynecology,Payments-Specialty
...,...,...,...
961916,1285655068,Pediatrics,Payments-Specialty
961927,1386870376,Internal Medicine,Payments-Specialty
961930,1972551679,Nuclear Medicine,Payments-Specialty
961937,1447211305,Urology,Payments-Specialty


In [30]:
gp_specialties_npi = gp_specialties_npi.rename(columns = {
        "covered_recipient_npi": "Rfrg_NPI",
        "specialty_lvl2": "Rfrg_Prvdr_Spclty_Desc",
        "Rfrg_Prvdr_Spclty_Srce": "Rfrg_Prvdr_Spclty_Srce"})

In [31]:
gp_specialties_npi

,Rfrg_NPI,Rfrg_Prvdr_Spclty_Desc,Rfrg_Prvdr_Spclty_Srce
0,1497734156,Orthopaedic Surgery,Payments-Specialty
799481,1275536377,Family Medicine,Payments-Specialty
799455,1609893270,Internal Medicine,Payments-Specialty
799456,1669440822,Internal Medicine,Payments-Specialty
799457,1003815317,Obstetrics & Gynecology,Payments-Specialty
...,...,...,...
961916,1285655068,Pediatrics,Payments-Specialty
961927,1386870376,Internal Medicine,Payments-Specialty
961930,1972551679,Nuclear Medicine,Payments-Specialty
961937,1447211305,Urology,Payments-Specialty


In [33]:
one_specialty_per_npi = pd.concat([dme_specialties_npi_sorted, gp_specialties_npi], ignore_index=True)

In [34]:
one_specialty_per_npi.drop_duplicates(subset = ["Rfrg_NPI"] , inplace = True, keep="first")

In [35]:
one_specialty_per_npi.shape

(884642, 3)

### Specialty Crosswalk

Merge the manually created specialty_crosswalk_gpt_backfill.csv

Directions on how to create the specialty_crosswalk_gpt_backfill.csv:
- This manual CSV was created by using the outputs of this script (=dmepos_specialties_unique and the gp_specialties_unique CSVs). It was manually deduplicated and ChatGPT was used to fill out the group levels. Then manual validation was applied to the new backfilled information. 

This is then merged to the one_specialty_per_npi_final by the og_specialty_name and Rfrg_Prvdr_Spclty_Desc to get the hierarchy decriptions. 

In [37]:
specialty_crosswalk = pd.read_csv("/dsa/groups/casestudycf25/team02/specialty_crosswalk_gpt_backfill.csv")

In [38]:
specialty_crosswalk.head()

,specialty_type,specialty_lvl1,specialty,og_specialty_name
0,Chiropractor,Chiropractic Providers,Chiropractor,Chiropractic
1,Chiropractor,Chiropractic Providers,Chiropractor,Chiropractor
2,Doctor of Dentistry,Dental Providers,Advanced Practice Dental Therapist,Advanced Practice Dental Therapist
3,Doctor of Dentistry,Dental Providers,Dental Assistant,Dental Assistant
4,Doctor of Dentistry,Dental Providers,Dental Hygienist,Dental Hygienist


In [41]:
one_specialty_per_npi_final = one_specialty_per_npi.merge(
    specialty_crosswalk,
    left_on = "Rfrg_Prvdr_Spclty_Desc",
    right_on = "og_specialty_name",
    how="inner",          
    validate="many_to_one" 
)
one_specialty_per_npi_final.shape

(884642, 7)

In [43]:
one_specialty_per_npi_final.drop(columns = ["Rfrg_Prvdr_Spclty_Desc", "og_specialty_name"], inplace =True)

In [46]:
one_specialty_per_npi_final

,Rfrg_NPI,Rfrg_Prvdr_Spclty_Srce,specialty_type,specialty_lvl1,specialty
0,1992997787,NPPES-Specialty,Other,Physician Assistants & Advanced Practice Nursi...,Physician Assistant
1,1144628355,NPPES-Specialty,Other,Physician Assistants & Advanced Practice Nursi...,Physician Assistant
2,1144656935,NPPES-Specialty,Other,Physician Assistants & Advanced Practice Nursi...,Physician Assistant
3,1144726290,NPPES-Specialty,Other,Physician Assistants & Advanced Practice Nursi...,Physician Assistant
4,1144777293,NPPES-Specialty,Other,Physician Assistants & Advanced Practice Nursi...,Physician Assistant
...,...,...,...,...,...
884637,1225689060,Payments-Specialty,Doctor of Dentistry,Dental Providers,Dental Therapist
884638,1043558984,Payments-Specialty,Doctor of Dentistry,Dental Providers,Dental Therapist
884639,1265726863,Payments-Specialty,Doctor of Dentistry,Dental Providers,Dental Laboratory Technician
884640,1356443675,Payments-Specialty,Medical Doctor,Allopathic & Osteopathic Physicians,Clinical Pharmacology


In [44]:
# Number of NaNs in the column 'covered_recipient_npi'
n_nans = one_specialty_per_npi_final['specialty'].isna().sum()
print("NaNs in specialty:", n_nans)
nan_percent = one_specialty_per_npi_final['specialty'].isna().mean() * 100
print(f"Percentage of NaNs in specialty: {nan_percent:.2f}%")

NaNs in specialty: 0
Percentage of NaNs in specialty: 0.00%


In [45]:
one_specialty_per_npi_final.to_csv("/dsa/groups/casestudycf25/team02/one_specialty_per_npi.csv")